In [ ]:
import pandas as pd
import numpy as np

true_df = pd.read_csv("../data/ibd_Truth.csv", sep = ",")

pred_df = pd.read_csv("../results/anopheles_ibd.ibd", sep = "\t", header = None)
pred_df.columns = ["id1", "id1 haplotype index", "id2", "id2 haplotype index", "chr", "start", "end", "LOD score"]
pred_df["length"] = pred_df["end"] - pred_df["start"]
pred_df = pred_df[pred_df["length"] > 2e4]
pred_df.head()

In [ ]:
BP_TO_CM_COEFF = 1 / 1000000


def calculate_segment_power(true_df, pred_df, BP_TO_CM_COEFF):
    
    pred_df['id1'] = pred_df['id1'].astype(str).str.replace('tsk_', '').astype(np.int64)
    pred_df['id2'] = pred_df['id2'].astype(str).str.replace('tsk_', '').astype(np.int64)
    
    power_results = []
    total_rows = len(true_df)
    
    for index, row in true_df.iterrows():
        if index % 10000 == 0:
            print(f"Calculating the progress: {index}/{total_rows}...")
        
        id1, id2 = row['id1'], row['id2']
        t_start, t_end = row['start'], row['end']
        
        bp_length = t_end - t_start
        cM_length = bp_length * BP_TO_CM_COEFF

        #1. Filter the predicted fragments from the prediction files.
        matches = pred_df[
            ((pred_df['id1'] == id1) & (pred_df['id2'] == id2) | 
             (pred_df['id1'] == id2) & (pred_df['id2'] == id1))
        ]
        
        total_overlap = 0

        #2. Calculate the overlap lengths
        for _, p_row in matches.iterrows():
            p_start, p_end = p_row['start'], p_row['end']
            
            overlap_start = max(t_start, p_start)
            overlap_end = min(t_end, p_end)
            
            if overlap_end > overlap_start:
                total_overlap += (overlap_end - overlap_start)

        segment_power = min(total_overlap / (t_end - t_start), 1.0)
        
        power_results.append({
            'True_Length_cM': cM_length,
            'Power': segment_power
        })
        
    return pd.DataFrame(power_results)



In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict


gt_sorted = true_df.sort_values(['id1', 'id2', 'start']).copy()

group_id = (gt_sorted['start'] > gt_sorted.groupby(['id1', 'id2'])['end'].shift().cummax()).groupby([gt_sorted['id1'], gt_sorted['id2']]).cumsum()

true_df_merged = gt_sorted.groupby(['id1', 'id2', group_id]).agg({
    'start': 'min',
    'end': 'max'
}).reset_index() 

true_df_merged['length'] = true_df_merged['end'] - true_df_merged['start']

print(f"raw truth segments {len(true_df)} -> non-overlapped truth segments {len(true_df_merged)}")

In [ ]:
# Power
print(f"successfully read! True segments: {len(true_df_merged)}, Predict segments: {len(pred_df)}")
print("Calculate segment-level Power...")
raw_power_df = calculate_segment_power(true_df_merged, pred_df, BP_TO_CM_COEFF)

bin_edges = [0, 1, 2, 3, 4, 5]
bin_labels = ['0-1', '1-2', '2-3', '3-4', '4-5']

raw_power_df['Length_Bin'] = pd.cut(raw_power_df['True_Length_cM'], 
                                    bins=bin_edges, 
                                    labels=bin_labels, 
                                    include_lowest=True)


summary_df = raw_power_df.groupby('Length_Bin', observed=False)['Power'].agg(['mean', 'count']).reset_index()
summary_df.columns = ['Length_Bin_cM', 'Mean_Power', 'Segment_Count']

print("\n======== Power report ========\n ")
print(summary_df)

In [ ]:
# Accuracy

def calculate_overlap(p_start, p_end, t_start, t_end):
    return max(0, min(p_end, t_end) - max(p_start, t_start))

print("\n=== Base-pair Level Precision Report ===\n")

pair_basepairs = defaultdict(lambda: [0, 0])

pred_df['id1_str'] = pred_df['id1'].astype(str).str.replace('tsk_', '')
pred_df['id2_str'] = pred_df['id2'].astype(str).str.replace('tsk_', '')

true_lookup = defaultdict(list)
for row in true_df_merged.itertuples(index=False):
    r_dict = row._asdict()
    sorted_pair = tuple(sorted([str(r_dict['id1']), str(r_dict['id2'])]))
    true_lookup[sorted_pair].append((r_dict['start'], r_dict['end']))

for row in pred_df.itertuples(index=False):
    p_dict = row._asdict()
    p_len = p_dict['end'] - p_dict['start']
    if p_len <= 0: 
        continue
        
    p_pair_key = tuple(sorted([p_dict['id1_str'], p_dict['id2_str']]))
    
    max_overlap = 0
    if p_pair_key in true_lookup:
        for t_start, t_end in true_lookup[p_pair_key]:
            
            overlap = max(0, min(p_dict['end'], t_end) - max(p_dict['start'], t_start))
            if overlap > max_overlap:
                max_overlap = overlap

    pair_basepairs[p_pair_key][0] += max_overlap  
    pair_basepairs[p_pair_key][1] += p_len     


pair_level_accuracies = []
for pair, (total_overlap, total_pred_len) in pair_basepairs.items():
    if total_pred_len > 0:
        pair_level_accuracies.append(float(total_overlap) / total_pred_len)

if len(pair_level_accuracies) > 0:
    print(f"Mean Overlap Accuracy: {np.mean(pair_level_accuracies) * 100:.2f}%")
    print(f"SD: {np.std(pair_level_accuracies) * 100:.2f}%")